# OneBox vs Blackrock: how much does synchronization fix?

The same signals are recorded twice: the 14 s sync burst, eye-x, eye-y, pupil and the
photodiode go to the OneBox (XA0–XA4) **and** to Blackrock. Each system has its own
clock, so the two copies drift apart. This notebook puts both on Blackrock (NSP) time
and checks how well they agree.

- **before**: OneBox seconds (`sample / obSampRate`) plus *one* constant offset that
  lines up the start. That is what you would have without synchronization, and its
  error is the clock drift.
- **after**: `onebox_to_blackrock` from `time_map.json`, stamped into the analog export.

Both are measured on the same pairs of events. The **14 s burst pulses** were held out
of the fit, so they are the honest test. The **1 Hz** edges were used to fit the map,
so they only show how well it fits. A **photodiode** that Blackrock also records gives
a second independent check.

**Needs:** `run_exporting_pipeline.py` run with `export_analog: true`, so
`export/*.analog.mat` exists and is on Blackrock time. Blackrock's copies come from the
session: the burst from `blackrock.sync_file` (the `.ns5`, by channel id), and eye, pupil
and photodiode from `blackrock.analog_file` (the `.ns2`, by **position** in
`blackrock.analog_channels`). The same comparison runs automatically in
`validate_remapping` and lands in `export/sync_report.txt`; this notebook is for looking.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import spikesorting as ss
from spikesorting._io import analog
from spikesorting._sync import compare, extract

# ---------------- edit these ----------------
SESSION = "session_template_1probe"          # the session config's name
MACHINE = {"darwin": "mac", "win32": "windows_rig"}.get(sys.platform, "hpc")
MINUTES = 5                                  # first / last N minutes to overlay
WORST_WINDOW_S = 1.0                         # half-width around the worst event
ZOOM_MS = 5.0                                # half-width of the zoom on that edge
MAX_POINTS = 20_000                          # per trace drawn

# OneBox channels whose rising edges are compared as events, beside the sync burst.
EVENT_CHANNELS = ["photodiode"]

## 1. Load both systems

In [ ]:
config = ss.load_session_config(SESSION, MACHINE)
_, npx = config.per_probe()[0]              # one OneBox per run: it sits with the first probe

onebox = analog.load_onebox_analog(analog.find_onebox_analog(npx.paths.export_for("neuropixels")))
print(f"{onebox.path.name}: {', '.join(onebox.names)}")
print(f"  {onebox.fs:.4f} Hz, {onebox.n_samples / onebox.fs / 60:.1f} min")
print(f"  onebox_to_blackrock: slope {onebox.slope:.9f} ({(onebox.slope - 1) * 1e6:+.2f} ppm), "
      f"intercept {onebox.intercept:.6f} s")

brk = config.blackrock
blackrock = [analog.open_blackrock_analog(               # the burst, from the .ns5, by id
    brk.sync_file, {"sync_burst": brk.burst_channel},
    gap_tolerance_ms=brk.gap_tolerance_ms, tolerance_s=config.alignment_tolerance_s, by="id",
)]
if brk.analog_file is not None:                           # behaviour, from the .ns2, by position
    blackrock.append(analog.open_blackrock_analog(
        brk.analog_file, brk.analog_channels,
        gap_tolerance_ms=brk.gap_tolerance_ms, tolerance_s=config.alignment_tolerance_s,
        by="position",
    ))
else:
    print("no blackrock.analog_file in the session: behaviour channels drawn from the OneBox only")
for source in blackrock:
    resolved = {n: f"id {s.channel_id} ({s.channel_name})" for n, s in source.streams.items()}
    print(f"{source.path.name}: {resolved}  |  {source.time_map.summary()}")

t_start, t_stop = analog.get_recording_overlap(onebox, blackrock)
print(f"both recording for {(t_stop - t_start) / 60:.1f} min")

## 2. Events seen by both systems

The sync burst and 1 Hz edges come from the edge files `extract_sync` wrote: Blackrock's
are already on the NSP clock, the OneBox's on its own. Photodiode events are detected
here from both traces: an event is the signal leaving its baseline, up or down (a
photodiode may rest high and dip for the target), with levels set from excerpts across
the whole session, separately per system. Pass `levels=` to override them.

In [ ]:
npx_sync, br_sync = npx.paths.sync_for("neuropixels"), config.paths.sync_for("blackrock")
events = {
    "14 s burst (not in fit)": (
        analog.read_edge_file(br_sync / f"{extract.BR_BURST}.txt"),
        analog.read_edge_file(npx_sync / f"{extract.NPX_BURST}.txt"),
    ),
    "1 Hz (used in fit)": (
        analog.read_edge_file(br_sync / f"{extract.BR_1HZ}.txt"),
        analog.read_edge_file(npx_sync / f"{extract.NPX_OBX_1HZ}.txt"),
    ),
}
# Which Blackrock file records each label (the burst is in the .ns5, the rest in the .ns2).
br_source = {name: b for b in reversed(blackrock) for name in b.streams}
for name in EVENT_CHANNELS:
    if name not in br_source:
        print(f"{name}: not in blackrock.analog_channels, skipped")
        continue
    obx_t, obx_levels = analog.detect_onebox_events(onebox, name)
    br_t, br_levels = analog.detect_blackrock_events(br_source[name], name)
    print(f"{name}: {obx_t.size} OneBox events ({obx_levels.summary()}), "
          f"{br_t.size} Blackrock events ({br_levels.summary()})")
    events[name] = (br_t, obx_t)

## 3. Quantify: the error before and after synchronization

In [ ]:
comparisons = [
    compare.compare_event_times(br, obx, onebox.slope, onebox.intercept, label=name)
    for name, (br, obx) in events.items()
]
rows = {}
for c in comparisons:
    row = {"n_pairs": c.n_pairs}
    for which in ("before", "after"):
        err_ms = np.abs(getattr(c, f"{which}_s")) * 1e3
        row[f"{which}_median_ms"] = np.median(err_ms)
        row[f"{which}_p99_ms"] = np.percentile(err_ms, 99)
        row[f"{which}_max_ms"] = err_ms.max()
    row["improvement_x"] = row["before_max_ms"] / row["after_max_ms"]
    row["unpaired_blackrock"] = c.n_blackrock - c.n_pairs
    row["unpaired_onebox"] = c.n_onebox - c.n_pairs
    rows[c.label] = row
table = pd.DataFrame(rows).T
table.round(4)

In [ ]:
# Error over the session, unsynchronized (left) and synchronized (right); worst pair ringed.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for which, ax in zip(("before", "after"), axes):
    for color, c in zip(plt.rcParams["axes.prop_cycle"].by_key()["color"], comparisons):
        err_ms = getattr(c, f"{which}_s") * 1e3
        t_min = (c.blackrock_s - c.blackrock_s[0]) / 60
        ax.scatter(t_min, err_ms, s=6, color=color, linewidths=0,
                   label=f"{c.label}: max {table.loc[c.label, f'{which}_max_ms']:.3g} ms")
        w = np.argmax(np.abs(err_ms))
        ax.scatter(t_min[w], err_ms[w], s=90, facecolors="none", edgecolors=color)
    ax.axhline(0, color="0.6", linewidth=0.8)
    ax.set_xlabel("time into the session (min)")
    ax.set_ylabel("OneBox - Blackrock (ms)")
    ax.legend(fontsize=8)
axes[0].set_title("before: OneBox seconds + one start offset")
axes[1].set_title("after: onebox_to_blackrock map")
fig.tight_layout()
plt.show()

In [ ]:
# Notebook-only helpers: slice a window of both copies and draw them. Plain numpy/matplotlib.

def scale01(y):
    """0..1 between the 1st and 99th percentile (min..max for a sparse pulse train)."""
    if y.size == 0:
        return y
    lo, hi = np.percentile(y, [1, 99])
    if hi <= lo:
        lo, hi = y.min(), y.max()
    return (y - lo) / (hi - lo) if hi > lo else y - lo


def envelope(t, y, max_points=MAX_POINTS):
    """Min/max per block, so a 10 ms pulse survives a 5-minute window."""
    step = int(np.ceil(y.size / (max_points // 2)))
    if step <= 1:
        return t, y
    n = y.size // step * step
    blocks = y[:n].reshape(-1, step)
    return np.repeat(t[:n:step], 2), np.column_stack([blocks.min(1), blocks.max(1)]).ravel()


def window(name, t0, t1, onebox_offset_s=None):
    """(t_br, y_br, t_obx, y_obx) on Blackrock time; None where a system lacks the channel.

    onebox_offset_s puts the OneBox on the unsynchronized axis, k / fs + offset.
    """
    out = [None, None, None, None]
    if name in br_source:
        b = br_source[name]
        s0, s1 = compare.blackrock_sample_range(t0, t1, b.segments)
        out[0:2] = envelope(b.time_map.apply(np.arange(s0, s1)),
                            scale01(analog.read_blackrock(b, name, s0, s1)))
    if name in onebox.index:
        if onebox_offset_s is None:
            k0, k1 = compare.onebox_sample_range(t0, t1, onebox.fs, onebox.slope,
                                                  onebox.intercept, onebox.n_samples)
            t = onebox.blackrock_time(k0, k1)
        else:
            k0 = max(int((t0 - onebox_offset_s) * onebox.fs), 0)
            k1 = min(int(np.ceil((t1 - onebox_offset_s) * onebox.fs)), onebox.n_samples)
            t = np.arange(k0, k1) / onebox.fs + onebox_offset_s
        out[2:4] = envelope(t, scale01(analog.read_onebox(onebox, name, k0, k1)))
    return out


def draw(axes, t0, t1, origin, marker=None, onebox_offset_s=None):
    """One channel per axis: Blackrock solid black, OneBox dashed orange."""
    for ax, name in zip(axes, onebox.names):
        t_br, y_br, t_obx, y_obx = window(name, t0, t1, onebox_offset_s)
        if t_br is not None:
            ax.plot(t_br - origin, y_br, color="black", lw=0.9, label="Blackrock")
        ax.plot(t_obx - origin, y_obx, color="tab:orange", lw=0.9, ls="--", label="OneBox")
        if marker is not None:
            ax.axvline(marker - origin, color="crimson", lw=0.8, alpha=0.6)
        ax.set_ylabel(name, rotation=0, ha="right", va="center")
        ax.set_yticks([])
    axes[-1].set_xlabel(f"Blackrock time - {origin:.3f} (s)")


def plot_window(t0, t1, title, origin, marker=None):
    fig, axes = plt.subplots(len(onebox.names), 1, sharex=True, squeeze=False,
                             figsize=(13, 1.3 * len(onebox.names) + 1))
    draw(axes[:, 0], t0, t1, origin, marker)
    axes[0, 0].legend(loc="upper right", fontsize=8, ncol=2)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

## 4. The largest unsynchronized offset, before vs after

This is where drift has pulled the two copies furthest apart, usually near the end.
On the left the OneBox trace runs late or early by that amount; on the right the map
has put it back on the Blackrock trace.

In [ ]:
ref = comparisons[0]                                  # the burst: never in a fit
i = np.argmax(np.abs(ref.before_s))
t_event = ref.blackrock_s[i]
half = max(2 * abs(ref.before_s[i]), 0.05)
print(f"largest offset before sync: {ref.before_s[i] * 1e3:+.2f} ms "
      f"at {(t_event - t_start) / 60:.1f} min; after: {ref.after_s[i] * 1e6:+.1f} us")

fig, axes = plt.subplots(len(onebox.names), 2, sharex=True, squeeze=False,
                         figsize=(14, 1.3 * len(onebox.names) + 1.2))
draw(axes[:, 0], t_event - half, t_event + half, t_event, t_event, onebox_offset_s=ref.offset_s)
draw(axes[:, 1], t_event - half, t_event + half, t_event, t_event)
axes[0, 0].set_title("before synchronization")
axes[0, 1].set_title("after synchronization")
axes[0, 1].legend(loc="upper right", fontsize=8)
fig.suptitle(f"largest unsynchronized offset ({ref.label})")
fig.tight_layout()
plt.show()

## 5. Synchronized signals: first and last minutes

Blackrock is solid black, OneBox dashed orange, both on Blackrock time and scaled 0–1
(the two record different units). Long windows are drawn as a min/max envelope, so
short pulses stay visible.

In [ ]:
plot_window(t_start, t_start + 60 * MINUTES, f"first {MINUTES} min, synchronized", t_start)
plot_window(t_stop - 60 * MINUTES, t_stop, f"last {MINUTES} min, synchronized",
            t_stop - 60 * MINUTES)

## 6. Where synchronization is worst

The single event with the largest error *after* the map, across every event set: first
the surrounding ±`WORST_WINDOW_S`, then a zoom on the edge itself.

In [ ]:
worst = max(comparisons, key=lambda c: np.abs(c.after_s).max())
j = np.argmax(np.abs(worst.after_s))
t_worst = worst.blackrock_s[j]
print(f"largest remaining asynchrony: {worst.after_s[j] * 1e3:+.4f} ms ({worst.label}) "
      f"at {(t_worst - t_start) / 60:.2f} min")

for half, label in ((WORST_WINDOW_S, f"+/-{WORST_WINDOW_S:g} s"),
                    (ZOOM_MS / 1e3, f"+/-{ZOOM_MS:g} ms")):
    plot_window(t_worst - half, t_worst + half,
                f"largest remaining asynchrony ({worst.label}), {label}", t_worst, t_worst)